# Discrete time models

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

### Binomial model

Simulate price $(S_k)_{k\ge 0}$ at Binomial model up to time $n$

In [ ]:
def binom_simulations(S0, u, d, p, time, n_paths):
    
    trajectories = []    
    for _ in range(n_paths):
        xs = random.choices([u, d], weights = [p, 1-p], k = time)
        xs_initial = [1] + xs
        prices = S0*np.cumprod(xs_initial)
        trajectories.append(prices)
        
    return np.array(trajectories)    

Verify that sample mean close to theoretical mean $\mathsf E(S_n)$.

In [ ]:
S0 = 100
u = 1.02
d = 0.99
p = 0.52
n = 15
N = 100_000
paths = binom_simulations(S0, u, d, p, n, N)

Sn = paths[:, -1]
mean = np.mean(Sn)

theor_mean = S0*(u*p+d*(1-p))**n
print(mean, theor_mean, abs(mean-theor_mean))

Find probability $P(\mu - 3 \Delta < S_n < mu + 3\Delta),\ \Delta = |\mu - median|$

In [ ]:
S0 = 100
u = 1.02
d = 0.99
p = 0.52
n = 15
N = 100_000
paths = binom_simulations(S0, u, d, p, n, N)

Sn = paths[:, -1]

delta = abs(np.mean(Sn) - np.median(Sn))
A = np.mean(Sn)- 3*delta
B = np.mean(Sn)+ 3*delta
# :)))  
Prob = sum((Sn > A) & (Sn < B))/N
Prob

Build confidence interval $CI_{1-\alpha}(S_n)$ 

In [ ]:
S0 = 100
u = 1.02
d = 0.99
p = 0.52
n = 15
N = 100_000
paths = binom_simulations(S0, u, d, p, n, N)
Sn = paths[:, -1]

interval = np.quantile(Sn, [0.025, 0.975])
interval                       

Investivate a time of reaching the level $A>0$, i.e. $\tau_A=min\{k\geq 0: S_k\geq A\}$.

In [ ]:
S0 = 100
u = 1.02
d = 0.99
p = 0.52
n = 15
N = 100_000

A = 108
paths = binom_simulations(S0, u, d, p, n, N)

tau = []
for path in paths:
    for i in range(n+1):
        if path[i] >= A:
            tau.append(i)
            break
        if i == n:
            tau.append(n+1)
            
plot = go.Figure()
hist = go.Histogram(x = tau, histnorm = 'probability density', nbinsx = 10)
plot.add_trace(hist)
plot.show()

### Random walk

In [ ]:
def rw_simulations(p, max_time, n_paths):
    paths = []
    for i in range(0, n_paths):
        bern = [0] + random.choices([-1, 1], weights = [1-p, p], k = max_time)
        path = np.cumsum(bern)
        paths.append(path)
    return np.array(paths)

In [ ]:
p = 0.53
n = 15
N = 3

paths = rw_simulations(p, n, N)

x_ = np.arange(n+1)
plot = go.Figure()
for i in range(N):
    graph = go.Scatter(x = x_, y = paths[i])
    plot.add_trace(graph)

plot.show()

Verify that answers $Var(R_n)=4p(1-p)n$ via paths simulation

In [ ]:
def variance_test(p, time, N, tolerance):
    paths = rw_simulations(p, time, N)
    Rn = paths[:, -1]
    var = np.var(Rn, ddof = 1)
    theor_var = 4*time*p*(1-p)
    
    return abs(var-theor_var) < tolerance

In [ ]:
p = 0.6
n = 15
N = 250_000

variance_test(p, n, N, 0.01)

<b> Expected number of 0's of simple symmetric Random walk ($p=1/2$).</b>

Take consecutively $n=5,10,15,\ldots,100$ and check number of 0's.

In [ ]:
p = 0.5
# Once you try 1000 try 5000 and check the difference
N = 1000
Expected_n_zeros = []
for n in range(5,101,5):
    paths = rw_simulations(p, n, N)
    zeros = [sum(path == 0) for path in paths]
    Expected_n_zeros.append(np.mean(zeros))

Actually, number of zeros behaves like $\sqrt{\frac{2}{\pi}n}$. Compare this with our result.

In [ ]:
ns = np.arange(5, 101, 5)
correct = np.sqrt(2*ns/np.pi)

plot = go.Figure()
graph1 = go.Scatter(x = ns, y = Expected_n_zeros, mode = 'markers', name = 'numerical')
graph2 = go.Scatter(x = ns, y = correct, name = 'theoretical')
plot.add_traces([graph1, graph2])
plot.show()

### Markov chains

**Question 1, midterm 2024-25 [17 pts]**. Consider Markov chain $(C_n)_{n\ge 0}$ with $C_0=0$ states $S=\{0, 2\}$ and the following transitional probabilities: $p_{0,0}=1/8,\,p_{0,2}=7/8,\, p_{2,0}=3/4,\,p_{2,2}=1/4$. Find final probabilities $\pi_0,\,\pi_2$ numerically.</br>
Output: numerical values of $\widehat{\pi_0},\,\widehat{\pi_2}$ based on distribution of $C_{100}$ using $N=1000$ simulations. If you found theoretical values of $\pi_0,\,\pi_2$ in 2c) additionally output $|\pi_0-\widehat{\pi_0}|,\,|\pi_2-\widehat{\pi_2}|$.

In [ ]:
def markov_chain(C0, time, n_paths):
    paths = []
    for i in range(n_paths):
        path = [C0]
        for _ in range(time):
            if path[-1] == 0:
                new_el = random.choices([0, 2], weights = [0.125, 0.875], k = 1)[0]
            if path[-1] == 2:
                new_el = random.choices([0, 2], weights = [0.75, 0.25], k = 1)[0]
            path.append(new_el)
            
        paths.append(np.array(path))
    return np.array(paths)

In [ ]:
n = 50
N = 1
C0 = 2

paths = markov_chain(C0, n, N)
x_ = np.arange(n+1)

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = x_, y = path)
    plot.add_trace(graph)
plot.show()

In [ ]:
C0 = 2
n = 180
N = 10_000

paths = markov_chain(C0, n, N)
Cn = paths[:, -1]
pi0 = sum(Cn == 0)/N
pi2 = sum(Cn == 2)/N
print(pi0, pi2, abs(pi0-6/13), abs(pi2 - 7/13))